In [ ]:
import ee
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from datetime import datetime

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Define Mumbai shapefile
mumbai = ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file")

# Fetch Sentinel-2 image
def fetch_sentinel2_image(start_date, end_date):
    collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterDate(start_date, end_date) \
        .filterBounds(mumbai) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
        .select(['B2', 'B3', 'B4', 'B8'])
    return collection.median().clip(mumbai)

# Fetch NO2 image
def fetch_no2_image(start_date, end_date):
    collection = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2") \
        .select('tropospheric_NO2_column_number_density') \
        .filterDate(start_date, end_date)
    return collection.mean().clip(mumbai)

# Sample training data
def sample_training_data(start_date, end_date, scale=10):
    sentinel2_image = fetch_sentinel2_image(start_date, end_date)
    no2_image = fetch_no2_image(start_date, end_date)

    # Combine Sentinel-2 bands and NO₂
    combined_image = sentinel2_image.addBands(no2_image)

    samples = combined_image.sample(
        region=mumbai,
        scale=scale,
        numPixels=5000,
        geometries=False
    ).getInfo()

    predictors = []
    target = []

    for feature in samples['features']:
        props = feature['properties']
        if all(band in props for band in ["B2", "B3", "B4", "B8"]) and 'tropospheric_NO2_column_number_density' in props:
            predictors.append([props["B2"], props["B3"], props["B4"], props["B8"]])
            target.append(props['tropospheric_NO2_column_number_density'])

    return np.array(predictors), np.array(target)

# Train the Linear Regression model
def train_model(predictors, target):
    scaler = StandardScaler()
    predictors = scaler.fit_transform(predictors)

    model = LinearRegression()
    model.fit(predictors, target)
    return model, scaler

if __name__ == "__main__":
    start_date = '2018-06-28'  # Fixed start date
    end_date = datetime.now().strftime('%Y-%m-%d')  # Dynamic end date

    print("Sampling training data...")
    predictors, target = sample_training_data(start_date, end_date, scale=100)

    print("Training the Linear Regression model...")
    model, scaler = train_model(predictors, target)

    print("Saving model and scaler...")
    with open('/content/NO2_lr_model.pkl', 'wb') as f:
        pickle.dump(model, f)
    np.save('/content/predictor_scaler_mean.npy', scaler.mean_)
    np.save('/content/predictor_scaler_scale.npy', scaler.scale_)

Sampling training data...
Training the Linear Regression model...
Saving model and scaler...


In [ ]:
import ee
import numpy as np
import pickle

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Load trained model coefficients and scaler values
with open('/content/NO2_lr_model.pkl', 'rb') as f:
    model = pickle.load(f)  # Linear regression model (coefficients & intercept)
scaler_mean = np.load('/content/predictor_scaler_mean.npy')
scaler_scale = np.load('/content/predictor_scaler_scale.npy')

# Define Mumbai shapefile
mumbai_geometry = ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file").geometry()

# Fetch image with fallbacks
def fetch_image(user_date):
    start_date = ee.Date(user_date)
    end_date = start_date.advance(7, 'day')

    def has_valid_bands(image):
        return image.bandNames().size().getInfo() > 0  # Check for valid bands

    # Sentinel-2
    s2_image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
        .select(["B2", "B3", "B4", "B8"]).median()

    if has_valid_bands(s2_image):
        print("Using Sentinel-2 data.")
        return s2_image.clip(mumbai_geometry)

    # Landsat-8 fallback
    print("No Sentinel-2 data. Trying Landsat-8.")
    landsat_image = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5']).median()

    if has_valid_bands(landsat_image):
        landsat_image = landsat_image.multiply(0.0000275).add(-0.2).rename(['B2', 'B3', 'B4', 'B8'])
        print("Using Landsat-8 data.")
        return landsat_image.clip(mumbai_geometry)

    # MODIS fallback
    print("No Landsat-8 data. Trying MODIS.")
    modis_image = ee.ImageCollection("MODIS/061/MCD43A4") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .select(['Nadir_Reflectance_Band1', 'Nadir_Reflectance_Band4', 'Nadir_Reflectance_Band3', 'Nadir_Reflectance_Band2']) \
        .median().rename(['B2', 'B3', 'B4', 'B8'])

    if has_valid_bands(modis_image):
        print("Using MODIS data.")
        return modis_image.clip(mumbai_geometry)

    # Historical Median Composite
    print("No valid data. Using historical median composite.")
    historical_image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(mumbai_geometry) \
        .filterDate("2018-01-01", "2024-11-30") \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
        .select(['B2', 'B3', 'B4', 'B8']).median()

    return historical_image

# Apply linear regression model
def apply_model(image):
    try:
        # Get model coefficients
        coefficients = model.coef_
        intercept = model.intercept_

        bands = ['B2', 'B3', 'B4', 'B8']

        # Initialize a zero-image
        prediction = ee.Image(0).clip(image.geometry())

        # Add scaled bands iteratively
        for i, band in enumerate(bands):
            scaled_band = image.select(band) \
                               .subtract(scaler_mean[i]) \
                               .divide(scaler_scale[i]) \
                               .multiply(coefficients[i])
            prediction = prediction.add(scaled_band)

        # Add intercept
        prediction = prediction.add(intercept)

        return prediction.rename('Downscaled_NO2')

    except Exception as e:
        print(f"Error in model application: {e}")
        return None

# Predict and export NO₂
def predict_no2(user_date):
    try:
        print(f"Processing downscaled NO₂ for {user_date}...")
        print("Fetching Sentinel-2 data...")
        image = fetch_image(user_date)

        print("Applying linear regression model...")
        downscaled_no2 = apply_model(image)
        if downscaled_no2 is None:
            raise ValueError("Model application failed. No valid NO₂ image generated.")

        # Export the downscaled NO₂ image
        print("Exporting downscaled NO₂...")
        task = ee.batch.Export.image.toAsset(
            image=downscaled_no2,
            description='Downscaled_NO2_LR',
            assetId='projects/ee-chandanagalgali/assets/downscaled_NO2_lr',
            region=mumbai_geometry,
            scale=10,
            maxPixels=1e13
        )
        task.start()
        print("Export task started! Monitor progress in Google Earth Engine Tasks.")

    except Exception as e:
        print(f"Error processing NO₂: {e}")

if __name__ == "__main__":
    user_date = input("Enter the date for prediction (YYYY-MM-DD): ")
    predict_no2(user_date)
    print("Processing complete.")

Enter the date for prediction (YYYY-MM-DD): 2023-12-10
Processing downscaled NO₂ for 2023-12-10...
Fetching Sentinel-2 data...
Using Sentinel-2 data.
Applying linear regression model...
Exporting downscaled NO₂...
Export task started! Monitor progress in Google Earth Engine Tasks.
Processing complete.


In [2]:
#for SO2
import ee
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from datetime import datetime

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Define Mumbai shapefile
mumbai = ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file")

# Fetch Sentinel-2 image
def fetch_sentinel2_image(start_date, end_date):
    collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterDate(start_date, end_date) \
        .filterBounds(mumbai) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
        .select(['B2', 'B3', 'B4', 'B8'])
    return collection.median().clip(mumbai)

# Fetch SO₂ image
def fetch_so2_image(start_date, end_date):
    collection = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_SO2") \
        .select('SO2_column_number_density') \
        .filterDate(start_date, end_date)
    return collection.mean().clip(mumbai)

# Sample training data
def sample_training_data(start_date, end_date, scale=10):
    sentinel2_image = fetch_sentinel2_image(start_date, end_date)
    so2_image = fetch_so2_image(start_date, end_date)

    # Combine Sentinel-2 bands and SO₂
    combined_image = sentinel2_image.addBands(so2_image)

    samples = combined_image.sample(
        region=mumbai,
        scale=scale,
        numPixels=5000,
        geometries=False
    ).getInfo()

    predictors = []
    target = []

    for feature in samples['features']:
        props = feature['properties']
        if all(band in props for band in ["B2", "B3", "B4", "B8"]) and 'SO2_column_number_density' in props:
            predictors.append([props["B2"], props["B3"], props["B4"], props["B8"]])
            target.append(props['SO2_column_number_density'])

    return np.array(predictors), np.array(target)

# Train the Linear Regression model
def train_model(predictors, target):
    scaler = StandardScaler()
    predictors = scaler.fit_transform(predictors)

    model = LinearRegression()
    model.fit(predictors, target)
    return model, scaler

if __name__ == "__main__":
    start_date = '2018-06-28'  # Fixed start date
    end_date = datetime.now().strftime('%Y-%m-%d')  # Dynamic end date

    print("Processing SO₂...")

    print("Sampling training data...")
    predictors, target = sample_training_data(start_date, end_date, scale=100)

    print("Training the Linear Regression model...")
    model, scaler = train_model(predictors, target)

    print("Saving model and scaler for SO₂...")
    model_path = '/content/SO2_lr_model.pkl'
    scaler_mean_path = '/content/SO2_scaler_mean.npy'
    scaler_scale_path = '/content/SO2_scaler_scale.npy'

    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    np.save(scaler_mean_path, scaler.mean_)
    np.save(scaler_scale_path, scaler.scale_)


EEException: Caller does not have required permission to use project ee-chandanagalgali. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam/project?project=ee-chandanagalgali and then retry. Propagation of the new permission may take a few minutes.

In [ ]:
#for CO
import ee
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from datetime import datetime

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Define Mumbai shapefile
mumbai = ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file")

# Fetch Sentinel-2 image
def fetch_sentinel2_image(start_date, end_date):
    collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterDate(start_date, end_date) \
        .filterBounds(mumbai) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
        .select(['B2', 'B3', 'B4', 'B8'])
    return collection.median().clip(mumbai)

# Fetch CO image
def fetch_co_image(start_date, end_date):
    collection = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_CO") \
        .select('CO_column_number_density') \
        .filterDate(start_date, end_date)
    return collection.mean().clip(mumbai)

# Sample training data
def sample_training_data(start_date, end_date, scale=10):
    sentinel2_image = fetch_sentinel2_image(start_date, end_date)
    co_image = fetch_co_image(start_date, end_date)

    # Combine Sentinel-2 bands and CO
    combined_image = sentinel2_image.addBands(co_image)

    samples = combined_image.sample(
        region=mumbai,
        scale=scale,
        numPixels=5000,
        geometries=False
    ).getInfo()

    predictors = []
    target = []

    for feature in samples['features']:
        props = feature['properties']
        if all(band in props for band in ["B2", "B3", "B4", "B8"]) and 'CO_column_number_density' in props:
            predictors.append([props["B2"], props["B3"], props["B4"], props["B8"]])
            target.append(props['CO_column_number_density'])

    return np.array(predictors), np.array(target)

# Train the Linear Regression model
def train_model(predictors, target):
    scaler = StandardScaler()
    predictors = scaler.fit_transform(predictors)

    model = LinearRegression()
    model.fit(predictors, target)
    return model, scaler

if __name__ == "__main__":
    start_date = '2018-06-28'  # Fixed start date
    end_date = datetime.now().strftime('%Y-%m-%d')  # Dynamic end date

    print("Processing CO...")

    print("Sampling training data...")
    predictors, target = sample_training_data(start_date, end_date, scale=100)

    print("Training the Linear Regression model...")
    model, scaler = train_model(predictors, target)

    print("Saving model and scaler for CO...")
    model_path = '/content/CO_lr_model.pkl'
    scaler_mean_path = '/content/CO_scaler_mean.npy'
    scaler_scale_path = '/content/CO_scaler_scale.npy'

    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    np.save(scaler_mean_path, scaler.mean_)
    np.save(scaler_scale_path, scaler.scale_)


In [ ]:
#DYNAMIC FOR NO2,SO2,CO,O3,CH4,HCHO
import ee
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from datetime import datetime

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Define Mumbai shapefile
mumbai = ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file")

# Sentinel-5P bands for pollutants
POLLUTANT_BANDS = {
    "NO2": {"collection": "COPERNICUS/S5P/OFFL/L3_NO2", "band": "tropospheric_NO2_column_number_density"},
    "SO2": {"collection": "COPERNICUS/S5P/OFFL/L3_SO2", "band": "SO2_column_number_density"},
    "CO": {"collection": "COPERNICUS/S5P/OFFL/L3_CO", "band": "CO_column_number_density"},
    "O3": {"collection": "COPERNICUS/S5P/OFFL/L3_O3", "band": "O3_column_number_density"},
    "CH4": {"collection": "COPERNICUS/S5P/OFFL/L3_CH4", "band": "CH4_column_volume_mixing_ratio_dry"},
    "HCHO": {"collection": "COPERNICUS/S5P/OFFL/L3_HCHO", "band": "tropospheric_HCHO_column_number_density"}
}

# Fetch Sentinel-2 image
def fetch_sentinel2_image(start_date, end_date):
    collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterDate(start_date, end_date) \
        .filterBounds(mumbai) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
        .select(['B2', 'B3', 'B4', 'B8'])
    return collection.median().clip(mumbai)

# Fetch selected pollutant image
def fetch_pollutant_image(pollutant, start_date, end_date):
    if pollutant not in POLLUTANT_BANDS:
        raise ValueError(f"Pollutant '{pollutant}' is not supported. Available options: {list(POLLUTANT_BANDS.keys())}")

    collection = POLLUTANT_BANDS[pollutant]["collection"]
    band = POLLUTANT_BANDS[pollutant]["band"]

    return ee.ImageCollection(collection).select(band).filterDate(start_date, end_date).mean().clip(mumbai), band

# Sample training data
def sample_training_data(pollutant, start_date, end_date, scale=10):
    sentinel2_image = fetch_sentinel2_image(start_date, end_date)
    pollutant_image, band = fetch_pollutant_image(pollutant, start_date, end_date)

    # Combine Sentinel-2 bands and the pollutant
    combined_image = sentinel2_image.addBands(pollutant_image)

    samples = combined_image.sample(
        region=mumbai,
        scale=scale,
        numPixels=5000,
        geometries=False
    ).getInfo()

    predictors = []
    target = []

    for feature in samples['features']:
        props = feature['properties']
        if all(band_name in props for band_name in ["B2", "B3", "B4", "B8"]) and band in props:
            predictors.append([props["B2"], props["B3"], props["B4"], props["B8"]])
            target.append(props[band])

    return np.array(predictors), np.array(target)

# Train the Linear Regression model
def train_model(predictors, target):
    scaler = StandardScaler()
    predictors = scaler.fit_transform(predictors)

    model = LinearRegression()
    model.fit(predictors, target)
    return model, scaler

if __name__ == "__main__":
    start_date = '2018-06-28'  # Fixed start date
    end_date = datetime.now().strftime('%Y-%m-%d')  # Dynamic end date

    print("Available pollutants:", ", ".join(POLLUTANT_BANDS.keys()))
    pollutant = input("Enter the pollutant to process (e.g., NO2, SO2, CO, O3, CH4, HCHO): ").strip().upper()

    print(f"Processing {pollutant}...")

    print("Sampling training data...")
    predictors, target = sample_training_data(pollutant, start_date, end_date, scale=100)

    print("Training the Linear Regression model...")
    model, scaler = train_model(predictors, target)

    print(f"Saving model and scaler for {pollutant}...")
    model_path = f'/content/{pollutant}_lr_model.pkl'
    scaler_mean_path = f'/content/{pollutant}_scaler_mean.npy'
    scaler_scale_path = f'/content/{pollutant}_scaler_scale.npy'

    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    np.save(scaler_mean_path, scaler.mean_)
    np.save(scaler_scale_path, scaler.scale_)


In [ ]:
#dynamic
import ee
import numpy as np
import pickle

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Define Mumbai geometry
mumbai_geometry = ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file").geometry()

# Pollutant configuration
POLLUTANT_CONFIG = {
    "NO2": {
        "model_path": "/content/NO2_lr_model.pkl",
        "scaler_mean_path": "/content/predictor_scaler_mean.npy",
        "scaler_scale_path": "/content/predictor_scaler_scale.npy",
        "export_asset_id": "projects/ee-chandanagalgali/assets/downscaled_NO2_lr",
        "output_band": "Downscaled_NO2"
    },
    "SO2": {
        "model_path": "/content/SO2_lr_model.pkl",
        "scaler_mean_path": "/content/so2_scaler_mean.npy",
        "scaler_scale_path": "/content/so2_scaler_scale.npy",
        "export_asset_id": "projects/ee-chandanagalgali/assets/downscaled_SO2_lr",
        "output_band": "Downscaled_SO2"
    },
    "CO": {
        "model_path": "/content/CO_lr_model.pkl",
        "scaler_mean_path": "/content/co_scaler_mean.npy",
        "scaler_scale_path": "/content/co_scaler_scale.npy",
        "export_asset_id": "projects/ee-chandanagalgali/assets/downscaled_CO_lr",
        "output_band": "Downscaled_CO"
    },
    "HCHO": {
        "model_path": "/content/HCHO_lr_model.pkl",
        "scaler_mean_path": "/content/hcho_scaler_mean.npy",
        "scaler_scale_path": "/content/hcho_scaler_scale.npy",
        "export_asset_id": "projects/ee-chandanagalgali/assets/downscaled_HCHO_lr",
        "output_band": "Downscaled_HCHO"
    },
    "CH4": {
        "model_path": "/content/CH4_lr_model.pkl",
        "scaler_mean_path": "/content/ch4_scaler_mean.npy",
        "scaler_scale_path": "/content/ch4_scaler_scale.npy",
        "export_asset_id": "projects/ee-chandanagalgali/assets/downscaled_CH4_lr",
        "output_band": "Downscaled_CH4"
    },
    "O3": {
        "model_path": "/content/O3_lr_model.pkl",
        "scaler_mean_path": "/content/o3_scaler_mean.npy",
        "scaler_scale_path": "/content/o3_scaler_scale.npy",
        "export_asset_id": "projects/ee-chandanagalgali/assets/downscaled_O3_lr",
        "output_band": "Downscaled_O3"
    }
}

# Load the model and scaler
def load_model_and_scaler(pollutant):
    try:
        config = POLLUTANT_CONFIG[pollutant]
        with open(config["model_path"], 'rb') as f:
            model = pickle.load(f)
        scaler_mean = np.load(config["scaler_mean_path"])
        scaler_scale = np.load(config["scaler_scale_path"])
        return model, scaler_mean, scaler_scale, config
    except KeyError:
        raise ValueError(f"Pollutant '{pollutant}' not supported. Available options: {list(POLLUTANT_CONFIG.keys())}")

# Fetch image with fallbacks
def fetch_image(user_date):
    start_date = ee.Date(user_date)
    end_date = start_date.advance(7, 'day')

    def has_valid_bands(image):
        return image.bandNames().size().getInfo() > 0  # Check for valid bands

    # Sentinel-2
    s2_image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
        .select(["B2", "B3", "B4", "B8"]).median()

    if has_valid_bands(s2_image):
        print("Using Sentinel-2 data.")
        return s2_image.clip(mumbai_geometry)

    # Landsat-8 fallback
    print("No Sentinel-2 data. Trying Landsat-8.")
    landsat_image = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5']).median()

    if has_valid_bands(landsat_image):
        landsat_image = landsat_image.multiply(0.0000275).add(-0.2).rename(['B2', 'B3', 'B4', 'B8'])
        print("Using Landsat-8 data.")
        return landsat_image.clip(mumbai_geometry)

    # MODIS fallback
    print("No Landsat-8 data. Trying MODIS.")
    modis_image = ee.ImageCollection("MODIS/061/MCD43A4") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .select(['Nadir_Reflectance_Band1', 'Nadir_Reflectance_Band4', 'Nadir_Reflectance_Band3', 'Nadir_Reflectance_Band2']) \
        .median().rename(['B2', 'B3', 'B4', 'B8'])

    if has_valid_bands(modis_image):
        print("Using MODIS data.")
        return modis_image.clip(mumbai_geometry)

    # Historical Median Composite
    print("No valid data. Using historical median composite.")
    historical_image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(mumbai_geometry) \
        .filterDate("2018-01-01", "2024-11-30") \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
        .select(['B2', 'B3', 'B4', 'B8']).median()

    return historical_image

# Apply linear regression model
def apply_model(image, model, scaler_mean, scaler_scale, output_band):
    try:
        # Get model coefficients
        coefficients = model.coef_
        intercept = model.intercept_

        bands = ['B2', 'B3', 'B4', 'B8']

        # Initialize a zero-image
        prediction = ee.Image(0).clip(image.geometry())

        # Add scaled bands iteratively
        for i, band in enumerate(bands):
            scaled_band = image.select(band) \
                               .subtract(scaler_mean[i]) \
                               .divide(scaler_scale[i]) \
                               .multiply(coefficients[i])
            prediction = prediction.add(scaled_band)

        # Add intercept
        prediction = prediction.add(intercept)

        return prediction.rename(output_band)

    except Exception as e:
        print(f"Error in model application: {e}")
        return None

# Predict and export pollutant
def predict_pollutant(pollutant, user_date):
    try:
        print(f"Processing downscaled {pollutant} for {user_date}...")
        model, scaler_mean, scaler_scale, config = load_model_and_scaler(pollutant)

        print("Fetching Sentinel-2 data...")
        image = fetch_image(user_date)

        print("Applying linear regression model...")
        downscaled_pollutant = apply_model(image, model, scaler_mean, scaler_scale, config["output_band"])
        if downscaled_pollutant is None:
            raise ValueError("Model application failed. No valid image generated.")

        # Export the downscaled pollutant image
        print(f"Exporting downscaled {pollutant}...")
        task = ee.batch.Export.image.toAsset(
            image=downscaled_pollutant,
            description=f'Downscaled_{pollutant}_LR',
            assetId=config["export_asset_id"],
            region=mumbai_geometry,
            scale=10,
            maxPixels=1e13
        )
        task.start()
        print("Export task started! Monitor progress in Google Earth Engine Tasks.")

    except Exception as e:
        print(f"Error processing {pollutant}: {e}")

if __name__ == "__main__":
    pollutant = input("Enter the pollutant to process (e.g., NO2, SO2, CO, HCHO, CH4, O3): ").strip().upper()
    user_date = input("Enter the date for prediction (YYYY-MM-DD): ")
    predict_pollutant(pollutant, user_date)
    print("Processing complete.")
